# MoodNote-AI — Fine-tune PhoBERT trên Google Colab

Pipeline phân loại cảm xúc tiếng Việt với **PhoBERT + UIT-VSMEC + ViGoEmotions**.

**Trước khi chạy:**
1. Vào `Runtime → Change runtime type → T4 GPU`
2. Vào `Runtime → Manage Secrets` → thêm secret key **`HF_TOKEN`** (token HuggingFace của bạn)
3. Thay `REPO_URL` ở Cell 3 bằng URL GitHub của bạn
4. Chạy `Runtime → Run all`

## Cell 1 — Kiểm tra GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Không tìm thấy GPU!\n"
        "Vào Runtime → Change runtime type → chọn T4 GPU rồi chạy lại."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU   : {gpu_name}")
print(f"VRAM  : {gpu_mem:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print("Sẵn sàng train!")

## Cell 2 — Mount Google Drive

Model và checkpoint sẽ được lưu vào Drive để không mất khi session kết thúc.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Tạo thư mục lưu trữ trong Drive
DRIVE_BASE = '/content/drive/MyDrive/MoodNote-AI'
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/best_model',  exist_ok=True)

print(f"Drive mounted. Thư mục lưu: {DRIVE_BASE}")

## Cell 3 — Clone repo & cài dependencies

> **Thay `REPO_URL`** bằng URL GitHub của bạn, ví dụ:
> `https://github.com/username/MoodNote-AI.git`

In [ ]:
import sys

REPO_URL  = 'https://github.com/ToanHuynh0201/MoodNote-AI.git'  # <-- ĐỔI Ở ĐÂY
REPO_DIR  = '/content/MoodNote-AI'

# Clone repo
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo đã tồn tại tại {REPO_DIR}, bỏ qua clone.")
    !cd {REPO_DIR} && git pull

# Thêm vào Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Cài dependencies
print("\nCài đặt dependencies...")
!pip install -r {REPO_DIR}/requirements.txt -q

# Cài YAKE cho keyword extraction (FR-13)
print("Cài đặt YAKE...")
!pip install yake -q

print("\nHoàn tất cài đặt!")

## Cell 4 — Cấu hình paths cho Colab

In [ ]:
import os

# --- Paths ---
REPO_DIR       = '/content/MoodNote-AI'
CONFIG_DIR     = f'{REPO_DIR}/configs'
RAW_DIR        = f'{REPO_DIR}/data/raw'
MERGED_DIR     = f'{REPO_DIR}/data/merged'
PROCESSED_DIR  = f'{REPO_DIR}/data/processed'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'
BEST_MODEL_DIR = f'{DRIVE_BASE}/best_model'

os.makedirs(RAW_DIR,       exist_ok=True)
os.makedirs(MERGED_DIR,    exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Tắt W&B ---
os.environ['WANDB_MODE']    = 'disabled'
os.environ['WANDB_SILENT']  = 'true'

print("Paths đã cấu hình:")
print(f"  Config     : {CONFIG_DIR}")
print(f"  Data raw   : {RAW_DIR}")
print(f"  Merged     : {MERGED_DIR}")
print(f"  Processed  : {PROCESSED_DIR}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Best model : {BEST_MODEL_DIR}")
print("  W&B        : disabled")

## Cell 5 — Download UIT-VSMEC

In [ ]:
import os
os.chdir(REPO_DIR)

from src.data.download_dataset import download_uit_vsmec

print("=" * 50)
print("Bước 1: Download UIT-VSMEC")
print("=" * 50)
download_uit_vsmec(output_dir=RAW_DIR)

import pandas as pd
for split in ['train', 'validation', 'test']:
    f = f'{RAW_DIR}/{split}.csv'
    df = pd.read_csv(f)
    print(f"  {split:12s}: {len(df):,} mẫu")

## Cell 5.2 — Download ViGoEmotions (gated dataset)

> **Trước khi chạy:** vào `Runtime → Manage Secrets` → thêm secret với key **`HF_TOKEN`** và value là HuggingFace token của bạn (lấy tại [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)).

In [ ]:
import os
os.chdir(REPO_DIR)

# Đọc HF token từ Colab Secrets (Runtime → Manage Secrets → thêm key "HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("HF token loaded from Colab Secrets.")
except Exception:
    HF_TOKEN = None
    print("Warning: Không lấy được token từ Colab Secrets.")
    print("Nếu download thất bại, thêm secret 'HF_TOKEN' trong Runtime → Manage Secrets.")

from src.data.download_vigoemotions import download_vigoemotions

print("\n" + "=" * 50)
print("Bước 2: Download ViGoEmotions")
print("=" * 50)
download_vigoemotions(output_dir=RAW_DIR, token=HF_TOKEN)

## Cell 5.3 — Merge UIT-VSMEC + ViGoEmotions

Gộp 2 dataset thành một:
- **train / validation**: VSMEC + ViGoEmotions (mapped + deduplicated)
- **test**: VSMEC only (giữ clean benchmark)

In [ ]:
import os
os.chdir(REPO_DIR)

from src.data.merge_datasets import main as merge_main

print("=" * 50)
print("Bước 3: Merge UIT-VSMEC + ViGoEmotions")
print("=" * 50)
merge_main(
    vsmec_dir=RAW_DIR,
    vigoemotions_dir=f'{RAW_DIR}/vigoemotions',
    output_dir=MERGED_DIR,
)

## Cell 5.4 — Preprocess merged dataset (word segmentation)

In [ ]:
import os
os.chdir(REPO_DIR)

from src.data.preprocess import preprocess_dataset

print("=" * 50)
print("Bước 4: Preprocess (word segmentation)")
print("=" * 50)
preprocess_dataset(
    input_dir=MERGED_DIR,
    output_dir=PROCESSED_DIR,
    config_path=f'{CONFIG_DIR}/model_config.yaml'
)

import pandas as pd
for split in ['train', 'validation', 'test']:
    f = f'{PROCESSED_DIR}/{split}.csv'
    df = pd.read_csv(f)
    print(f"  {split:12s}: {len(df):,} mẫu — {f}")

## Cell 5.5 — Preprocess VSMEC-only (Curriculum Stage 1)

Tạo dataset VSMEC-only đã word-segment để train Stage 1 (clean data foundation trước khi học ViGoEmotions).

In [ ]:
import os
os.chdir(REPO_DIR)

from src.data.preprocess import preprocess_dataset

VSMEC_ONLY_DIR = f'{REPO_DIR}/data/processed/vsmec_only'
os.makedirs(VSMEC_ONLY_DIR, exist_ok=True)

print('=' * 50)
print('Bước 5.6: Preprocess VSMEC-only (curriculum Stage 1)')
print('=' * 50)
preprocess_dataset(
    input_dir=RAW_DIR,
    output_dir=VSMEC_ONLY_DIR,
    config_path=f'{CONFIG_DIR}/model_config.yaml'
)

import pandas as pd
df = pd.read_csv(f'{VSMEC_ONLY_DIR}/train.csv')
print(f'VSMEC-only train: {len(df):,} mẫu')
print(df['label'].value_counts().sort_index())

## Cell 5.6 — Data Augmentation (minority classes)

Augment các class thiếu mẫu (Anger, Fear, Disgust, Surprise) bằng random deletion, random swap và random insertion để tạo diverse synthetic samples và cân bằng dataset trước khi train.

In [ ]:
import os
import pandas as pd
os.chdir(REPO_DIR)

from src.data.augment import augment_dataset

AUGMENTED_TRAIN = f'{PROCESSED_DIR}/train_augmented.csv'

print("=" * 50)
print("Bước 5.6: Data Augmentation (minority classes)")
print("=" * 50)

# Xem phân phối hiện tại trước khi augment
train_df = pd.read_csv(f'{PROCESSED_DIR}/train.csv')
print("Phân phối trước augmentation:")
EMOTION_NAMES = {0: "Enjoyment", 1: "Sadness", 2: "Anger", 3: "Fear", 4: "Disgust", 5: "Surprise", 6: "Other"}
for label_idx, count in sorted(train_df['label'].value_counts().items()):
    print(f"  {EMOTION_NAMES[label_idx]:<12}: {count}")

# Augmentation targets (căn chỉnh theo kết quả test run 2):
#   Anger   (2): recall=0.35 → target 1800 (cần nhiều hơn để học ranh giới với Disgust)
#   Fear    (3): ~818 mẫu sau merge, thấp hơn median (~1100) → target 1400
#   Disgust (4): recall=0.57 → target 1300 (giảm để tránh lấn Anger)
#   Surprise(5): recall=0.30 → target 2000 (class yếu nhất, ít sample nhất)
#   Dùng 3 kỹ thuật: deletion + swap + insertion (đa dạng hóa augmented samples)
augment_dataset(
    input_csv=f'{PROCESSED_DIR}/train.csv',
    output_csv=AUGMENTED_TRAIN,
    target_counts={2: 1800, 3: 1400, 4: 1300, 5: 2000},
    techniques=["deletion", "swap", "insertion"],
    seed=42
)

## Cell 5.7 — Back-Translation Augmentation cho Anger

Anger (class 2) bị nhầm với Disgust nhiều nhất (40% confusion). Random deletion/swap không tạo đủ đa dạng từ vựng. Back-translation (VI→EN→VI) tạo paraphrase thực sự khác biệt.

> **Lưu ý:** Cell này gọi Google Translate API (~300 requests, ~30-60 giây với sleep 0.1s). Chạy sau Cell 5.6.

In [ ]:
import os
import time
import pandas as pd
os.chdir(REPO_DIR)

# Cài deep_translator (nhẹ, không cần restart)
!pip install deep_translator -q

from src.data.augment import VietnameseAugmenter

AUGMENTED_TRAIN = f'{PROCESSED_DIR}/train_augmented.csv'

print("=" * 50)
print("Bước 5.7: Back-Translation Augmentation (Anger)")
print("=" * 50)

# Lấy Anger samples từ processed train (trước augmentation)
# Dùng raw processed để tránh duplicate với augmented samples đã có
train_df = pd.read_csv(f'{PROCESSED_DIR}/train.csv')
anger_df = train_df[train_df['label'] == 2].reset_index(drop=True)
print(f"Anger samples gốc: {len(anger_df)}")

# Giới hạn 300 samples để tránh rate limit (đủ để tạo ~200-270 valid paraphrases)
anger_sample = anger_df.head(300)

aug = VietnameseAugmenter(seed=42)
bt_rows = []
failed = 0

print(f"Đang back-translate {len(anger_sample)} Anger samples...")
for i, (_, row) in enumerate(anger_sample.iterrows()):
    bt_text = aug.back_translate(row['text'])
    if bt_text != row['text'] and bt_text.strip():  # giữ nếu thực sự khác
        bt_rows.append({'text': bt_text, 'label': 2})
    else:
        failed += 1
    time.sleep(0.1)  # ~10 req/s để tránh rate limit
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/300 done — {len(bt_rows)} valid paraphrases")

print(f"\nBack-translation hoàn tất:")
print(f"  Valid paraphrases : {len(bt_rows)}")
print(f"  Skipped (same/err): {failed}")

if bt_rows:
    bt_df = pd.DataFrame(bt_rows)
    augmented_df = pd.read_csv(AUGMENTED_TRAIN)
    final_df = pd.concat([augmented_df, bt_df], ignore_index=True).sample(frac=1, random_state=42)
    final_df.to_csv(AUGMENTED_TRAIN, index=False)
    anger_total = len(final_df[final_df['label'] == 2])
    print(f"  Anger total sau BT: {anger_total}")
    print(f"  Dataset tổng      : {len(final_df)} mẫu")
    print(f"Đã lưu: {AUGMENTED_TRAIN}")
else:
    print("Không có paraphrase nào được tạo — bỏ qua merge.")

## Cell 6 — Train model (Curriculum Learning)

**Stage 1:** Train trên VSMEC-only (clean data, **10 epochs**) → xây dựng foundation tốt hơn.

**Stage 2:** Tiếp tục từ weights Stage 1 trên **augmented merged data** (20 epochs, LR=2e-5, patience=7).

Tổng thời gian ước tính: **70-90 phút** trên T4 GPU (PhoBERT-base).

In [ ]:
import os, torch, copy
import pandas as pd
os.chdir(REPO_DIR)

from src.data.dataset import EmotionDataset
from src.models.phobert_classifier import PhoBERTEmotionClassifier
from src.models.model_utils import save_model, get_device, print_model_summary
from src.training.trainer import train_model
from src.utils.config import load_all_configs
from src.utils.logger import setup_logger
from src.utils.metrics import compute_metrics, print_metrics, plot_confusion_matrix
import numpy as np
from pathlib import Path

logger = setup_logger(name='colab_training')

# Load configs
configs         = load_all_configs(CONFIG_DIR)
model_config    = configs['model']
training_config = configs['training']

logger.info(f"Model         : {model_config['model']['name']}")
logger.info(f"Epochs        : {training_config['training']['num_epochs']}")
logger.info(f"Batch size    : {training_config['training']['batch_size']}")
logger.info(f"LR            : {training_config['training']['learning_rate']}")
logger.info(f"Focal gamma   : {model_config['model'].get('focal_gamma', 2.0)}")

device = get_device()

# Paths
model_name      = model_config['model']['name']
max_len         = model_config['model']['max_seq_length']
VSMEC_ONLY_DIR  = f'{REPO_DIR}/data/processed/vsmec_only'
AUGMENTED_TRAIN = f'{PROCESSED_DIR}/train_augmented.csv'
STAGE1_DIR      = f'{CHECKPOINT_DIR}/stage1'

# Datasets — Stage 2 dùng augmented train
train_dataset = EmotionDataset(AUGMENTED_TRAIN,                   tokenizer_name=model_name, max_length=max_len)
val_dataset   = EmotionDataset(f'{PROCESSED_DIR}/validation.csv', tokenizer_name=model_name, max_length=max_len)
test_dataset  = EmotionDataset(f'{PROCESSED_DIR}/test.csv',       tokenizer_name=model_name, max_length=max_len)
vsmec_train   = EmotionDataset(f'{VSMEC_ONLY_DIR}/train.csv',     tokenizer_name=model_name, max_length=max_len)

logger.info(f"Train (augmented): {len(train_dataset)}, VSMEC-only: {len(vsmec_train)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Class weights — tính từ augmented train để phản ánh distribution mới
train_labels  = pd.read_csv(AUGMENTED_TRAIN)['label'].tolist()
class_counts  = np.bincount(train_labels, minlength=7).astype(float)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * 7
print("Class weights (từ augmented train):")
for i, w in enumerate(class_weights):
    print(f"  {model_config['emotion_labels'][i]:<12}: {w:.3f}")

# Model
model = PhoBERTEmotionClassifier(
    model_name=model_name,
    num_labels=model_config['model']['num_labels'],
    dropout=model_config['model']['dropout'],
    class_weights=class_weights,
    label_smoothing=model_config['model'].get('label_smoothing', 0.0),
    focal_gamma=model_config['model'].get('focal_gamma', 2.0)
)
model.to(device)
print_model_summary(model)

# ── Curriculum Stage 1: VSMEC-only (clean foundation) ─────────────────────────
print("\n" + "=" * 50)
print("Stage 1: VSMEC-only — 10 epochs (clean data)")
print("=" * 50)

stage1_config = copy.deepcopy(training_config)
stage1_config['training']['num_epochs']    = 10
stage1_config['training']['warmup_ratio']  = 0.2
stage1_config['training']['learning_rate'] = 2e-5  # Stage 1: LR cao hơn, data sạch hội tụ nhanh

train_model(
    model=model,
    train_dataset=vsmec_train,
    eval_dataset=val_dataset,
    training_config=stage1_config,
    output_dir=STAGE1_DIR,
    use_wandb=False
)

# ── Curriculum Stage 2: Augmented merged data ─────────────────────────────────
print("\n" + "=" * 50)
print("Stage 2: Augmented merged data — 20 epochs (from Stage 1 weights)")
print("=" * 50)
# training_config dùng LR=1e-5 (thấp hơn Stage 1) để tránh overfit trên augmented data

trainer = train_model(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    training_config=training_config,
    output_dir=CHECKPOINT_DIR,
    use_wandb=False
)

# Evaluate on test set
logger.info("Evaluating on test set...")
predictions = trainer.predict(test_dataset)
test_preds  = predictions.predictions
test_labels = predictions.label_ids

detailed = compute_metrics(test_preds, test_labels)
print_metrics(detailed, model_config['emotion_labels'])

# Confusion matrix
cm_path = Path(CHECKPOINT_DIR) / 'confusion_matrix.png'
plot_confusion_matrix(test_preds, test_labels,
                      emotion_labels=model_config['emotion_labels'],
                      save_path=cm_path)

# Save best model vào Drive
save_model(
    model=trainer.model,
    tokenizer=train_dataset.tokenizer,
    save_dir=BEST_MODEL_DIR,
    config={
        'model_config': model_config,
        'training_config': training_config,
        'test_results': {
            'accuracy':    detailed['accuracy'],
            'f1_macro':    detailed['f1_macro'],
            'f1_weighted': detailed['f1_weighted']
        }
    }
)

print("\n" + "=" * 50)
print("TRAINING HOÀN TẤT")
print("=" * 50)
print(f"Accuracy   : {detailed['accuracy']:.4f}")
print(f"F1-Macro   : {detailed['f1_macro']:.4f}")
print(f"F1-Weighted: {detailed['f1_weighted']:.4f}")
print(f"Model đã lưu tại: {BEST_MODEL_DIR}")

## Cell 7 — Kiểm tra model & test predict

In [ ]:
import os
os.chdir(REPO_DIR)

from src.models.model_utils import load_model
from src.inference.predictor import EmotionPredictor

# Kiểm tra file đã lưu trong Drive
print("Files trong best_model:")
for f in sorted(os.listdir(BEST_MODEL_DIR)):
    size = os.path.getsize(f'{BEST_MODEL_DIR}/{f}') / 1024**2
    print(f"  {f:<30} {size:.1f} MB")

# Test predict
print("\nTest predict:")
predictor = EmotionPredictor(model_path=BEST_MODEL_DIR)

test_sentences = [
    "Hôm nay tôi rất vui vì được nghỉ học!",
    "Tôi buồn quá, không biết phải làm sao.",
    "Thật tức giận khi bị đối xử bất công.",
    "Trời ơi, tin này làm tôi bất ngờ quá!",
]

for sentence in test_sentences:
    result = predictor.predict(sentence)
    print(f"  Text      : {sentence}")
    print(f"  Emotion   : {result['emotion']}")
    print(f"  Confidence: {result['confidence']:.2%}")
    print()